# Chat delay and clip alignment on the pilot VODs (#36)

Exploration notebook for issue #36. It reruns the study of
[`experiments/exp-001-chat-lag-alignment`](../experiments/exp-001-chat-lag-alignment/README.md)
on the 25 active pilot VODs and plots it. **Reported numbers come from the experiment's
`summary.json`, not from this notebook** (docs/ml/experiments.md).

How to run it:

1. `uv sync --group notebooks`
2. Point `AUTOCLIPPER_DATA_DIR` (in `.env` at the repository root) at the pilot data directory.
3. Run every cell, top to bottom. The first run builds the per-second series cache under
   `<artifacts>/cache/alignment/` (a few minutes); later runs take seconds.

Everything shown is counts and averages. Clear the outputs before committing (`nbstripout`
does it in pre-commit). Only VODs in `configs/pilot_vods.txt` are read; the historical
zero-clip VOD is excluded by the config.

All times are seconds of VOD time. Signals are per-second z-scores after removing a 5-minute
moving average: chat is `log1p` of messages per second (clip-link messages removed), loudness
is the loudest 64 ms frame of each second in dBFS. Profile curves are smoothed over 5 s.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

REPO = Path.cwd().resolve()
while not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
os.chdir(REPO)  # settings read `.env` from the working directory
sys.path.insert(0, str(REPO / "scripts"))

from run_pilot import read_vod_list  # noqa: E402

from autoclipper.config import get_settings  # noqa: E402
from autoclipper.features.chat import load_lexicon  # noqa: E402
from autoclipper.labels import alignment_study as study  # noqa: E402

config = study.StudyConfig.from_yaml(REPO / "experiments/exp-001-chat-lag-alignment/config.yaml")
settings = get_settings()
video_ids = read_vod_list(REPO / config.vod_list)
assert not set(video_ids) & set(config.exclude_video_ids), "the list names an excluded VOD"
len(video_ids)

In [ ]:
vods = study.load_collection(
    settings.data_dir,
    video_ids,
    lexicon=load_lexicon(REPO / config.lexicon),
    cache_dir=settings.artifacts_dir / "cache" / "alignment",
)
result = study.run_study(vods, config)
summary = study.rounded(result.summary)
summary["inputs"]

Plot style: the reference palette (fixed channel order), hairline grid, one y-axis.

In [ ]:
INK, INK_2, MUTED, GRID, AXIS, SURFACE = (
    "#0b0b0b",
    "#52514e",
    "#898781",
    "#e1e0d9",
    "#c3c2b7",
    "#fcfcfb",
)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
CHAT, LOUD = SERIES[0], SERIES[1]
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": AXIS,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "legend.frameon": False,
        "legend.labelcolor": INK_2,
        "lines.linewidth": 2,
        "figure.dpi": 110,
    }
)
CHANNELS = list(summary["inputs"]["channels"])
COLOR = dict(zip(CHANNELS, SERIES, strict=True))


def profile(signal, anchor, group="all", clip_class="all"):
    return result.profiles.filter(
        (pl.col("signal") == signal)
        & (pl.col("anchor") == anchor)
        & (pl.col("group") == group)
        & (pl.col("clip_class") == clip_class)
    ).sort("offset_s")


def draw(ax, frame, color, label, band=True):
    x = frame["offset_s"].to_numpy()
    if band:
        ax.fill_between(
            x, frame["low"].to_numpy(), frame["high"].to_numpy(), color=color, alpha=0.15, lw=0
        )
    ax.plot(x, frame["mean"].to_numpy(), color=color, label=label)


def mark_zero(ax, text):
    ax.axvline(0, color=INK_2, lw=1, ls="--")
    ax.annotate(
        text,
        (0, 1),
        xycoords=("data", "axes fraction"),
        xytext=(4, -4),
        textcoords="offset points",
        va="top",
        color=INK_2,
        fontsize=8,
    )


def peak_table(signal, anchor, clip_class="all"):
    rows = []
    for group, s in summary["profiles"][signal][anchor][clip_class].items():
        if s is None:
            continue
        rows.append(
            {
                "group": group,
                **{
                    k: s.get(k)
                    for k in (
                        "peak_offset_s",
                        "peak_low_s",
                        "peak_high_s",
                        "peak_height_z",
                        "chance_peak_height_p95_z",
                        "half_height_from_s",
                        "half_height_to_s",
                        "n_anchors",
                    )
                },
            }
        )
    return pl.DataFrame(rows)

## 1. Chat delay: chat around the streamer's loudest moments

Anchors are *loud events* (each VOD's loudest 0.5 % of seconds, at least 60 s apart), so no
clips are involved. Bands are 95 % intervals from resampling VODs. The table gives each
group's peak, its interval, the random-anchor chance level and where the curve stays above
half its height.

**Deviation from #36.** The issue asks for the delay *from chat peaks vs. clip starts*. Clip positions turned out to be clip ends (§ 2, exp-002/003), so clip starts can't time an event; the delay here is measured around loud moments instead, which needs no clips. Chat around clip starts, reported and corrected, is in § 2.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
draw(ax, profile("loudness", "loud_event"), LOUD, "loudness")
draw(ax, profile("chat", "loud_event"), CHAT, "chat")
delay = summary["chat_delay"]["loud_event"]["all"]
ax.plot(
    [delay["peak_offset_s"]], [delay["peak_height_z"]], "o", color=CHAT, ms=6, mec=SURFACE, mew=2
)
ax.annotate(
    f"chat peak +{delay['peak_offset_s']} s",
    (delay["peak_offset_s"], delay["peak_height_z"]),
    xytext=(8, 4),
    textcoords="offset points",
    color=INK,
    fontsize=9,
)
mark_zero(ax, "loud event")
ax.set(
    title=f"Chat after a loud event, all {summary['inputs']['n_vods']} VODs "
    f"({summary['inputs']['n_loud_events']} events)",
    xlabel="seconds from the loud event",
    ylabel="z-score",
)
ax.legend(loc="upper left")
plt.show()
peak_table("chat", "loud_event")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for login in CHANNELS:
    draw(ax, profile("chat", "loud_event", login), COLOR[login], login, band=False)
mark_zero(ax, "loud event")
ax.set(
    title="Chat after a loud event, per channel",
    xlabel="seconds from the loud event",
    ylabel="chat z-score",
)
ax.legend(ncols=5, loc="upper left", fontsize=8)
plt.show()

The whole-VOD cross-correlation of chat with loudness, per VOD (thin) and averaged
(thick). Its peaks are low; on talk-heavy channels loudness carries little event signal.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
for video_id in result.xcorr["video_id"].unique(maintain_order=True):
    one = result.xcorr.filter(pl.col("video_id") == video_id)
    ax.plot(one["lag_s"], one["corr"], color=AXIS, lw=0.8)
mean = result.xcorr.group_by("lag_s").agg(pl.col("corr").mean()).sort("lag_s")
ax.plot(mean["lag_s"], mean["corr"], color=CHAT, label="mean over VODs")
mark_zero(ax, "same second")
ax.set(
    title="Chat vs loudness over the whole VOD",
    xlabel="chat lag behind loudness (s)",
    ylabel="correlation",
)
ax.legend(loc="upper left")
plt.show()
pl.DataFrame([{"group": g, **v} for g, v in summary["chat_delay"]["whole_vod_xcorr"].items()])

## 2. Where clips sit: chat and loudness around clip positions

Zero is the clip's **reported** position (Helix `vod_offset`). Left: the start of each moment
(merged clips). Right: the start of each VOD's 5 most-viewed clips.

> ⚠️ **Reported positions are clip ends.** Zero in the first plots is Helix `vod_offset`, which exp-002 and exp-003 showed is where a clip *ends*, not where it starts (true start ≈ `vod_offset − duration`). Read against it, chat and loudness seem to come before the clip. The plots at the end of this section use corrected positions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, anchor, title in (
    (axes[0], "moment_start", "Around moment starts"),
    (axes[1], "top_clip_start", "Around top-clip starts"),
):
    draw(ax, profile("loudness", anchor), LOUD, "loudness")
    draw(ax, profile("chat", anchor), CHAT, "chat")
    mark_zero(ax, "reported start")
    n = summary["profiles"]["chat"][anchor]["all"]["all"]["n_anchors"]
    ax.set(title=f"{title} (n = {n})", xlabel="seconds from the reported start")
axes[0].set_ylabel("z-score")
axes[0].legend(loc="upper left")
plt.show()
pl.concat(
    [
        peak_table(s, a)
        .filter(pl.col("group") == "all")
        .with_columns(signal=pl.lit(s), anchor=pl.lit(a))
        for s in ("chat", "loudness")
        for a in ("moment_start", "top_clip_start", "top_clip_end")
    ]
)

Live clips against clips made after the stream (from the VOD), around moment starts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, signal in zip(axes, ("chat", "loudness"), strict=True):
    for clip_class, color in (("live", SERIES[0]), ("post_stream", SERIES[1])):
        n = summary["profiles"][signal]["moment_start"][clip_class]["all"]["n_anchors"]
        draw(
            ax,
            profile(signal, "moment_start", clip_class=clip_class),
            color,
            f"{clip_class} (n = {n})",
        )
    mark_zero(ax, "reported start")
    ax.set(
        title=f"{signal.capitalize()} around moment starts",
        xlabel="seconds from the reported start",
    )
axes[0].set_ylabel("z-score")
axes[0].legend(loc="upper left", fontsize=8)
plt.show()

Chat and loudness around **corrected** clip starts: every clip moved back by its own length plus its class's median end residual (the rule exp-002 recorded and applied in its rerun), then merged into moments again. Zero is now where the clip's content starts.

In [ ]:
import json

from autoclipper.labels import clip_offsets as co

exp002 = json.loads(
    (REPO / "experiments/exp-002-clip-offset-check/summary.json").read_text(encoding="utf-8")
)
rule = exp002["rerun"]["rule"]
moved = [co.shift_vod(v, rule["shift_s"], per_duration=rule["per_duration"]) for v in vods]
corrected = study.run_study(moved, config)


def corrected_profile(signal, anchor):
    return corrected.profiles.filter(
        (pl.col("signal") == signal)
        & (pl.col("anchor") == anchor)
        & (pl.col("group") == "all")
        & (pl.col("clip_class") == "all")
    ).sort("offset_s")


fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, anchor, title in (
    (axes[0], "moment_start", "Around corrected moment starts"),
    (axes[1], "top_clip_start", "Around corrected top-clip starts"),
):
    draw(ax, corrected_profile("loudness", anchor), LOUD, "loudness")
    draw(ax, corrected_profile("chat", anchor), CHAT, "chat")
    mark_zero(ax, "true start")
    n = corrected.summary["profiles"]["chat"][anchor]["all"]["all"]["n_anchors"]
    ax.set(title=f"{title} (n = {n})", xlabel="seconds from the corrected start")
axes[0].set_ylabel("z-score")
axes[0].legend(loc="upper left")
plt.show()
pl.DataFrame(
    [
        {"signal": s, "anchor": a, **corrected.summary["profiles"][s][a]["all"]["all"]}
        for s in ("chat", "loudness")
        for a in ("moment_start", "top_clip_start")
    ]
).select("signal", "anchor", "peak_offset_s", "peak_low_s", "peak_high_s", "n_anchors")

**Clip timing.** A clip made during the stream can only hold what has already happened, so
its content should end at or before its creation time. Below: live clips' creation time minus
their reported end, on the VOD timeline (VOD `created_at` taken as second 0). Negative means
created *before* the reported end.

In [ ]:
frames = []
for v in vods:
    start = v.clips["ended_at"] - pl.duration(seconds=v.duration_s)
    frames.append(
        v.clips.select(
            "duration_s",
            (
                (pl.col("created_at") - start).dt.total_milliseconds() / 1000
                - pl.col("vod_offset_s")
                - pl.col("duration_s")
            ).alias("created_after_end_s"),
            ((pl.col("created_at") - start).dt.total_milliseconds() / 1000 <= v.duration_s).alias(
                "live"
            ),
        )
    )
timing = pl.concat(frames).filter(pl.col("live"))
fig, ax = plt.subplots(figsize=(8, 3.4))
bins = np.arange(-90, 121, 5)
values = [
    timing.filter(pl.col("duration_s") == length)["created_after_end_s"].to_numpy()
    for length in (30.0, 60.0)
]
ax.hist(
    [np.clip(v, bins[0], bins[-1]) for v in values],
    bins=bins,
    color=SERIES[:2],
    stacked=True,
    label=[
        f"{int(length)} s clips (n = {v.size})" for length, v in zip((30, 60), values, strict=True)
    ],
    edgecolor=SURFACE,
    linewidth=1,
)
ax.grid(axis="x", visible=False)
mark_zero(ax, "reported end")
ax.set(
    title="Live clips: created minus reported end",
    xlabel="seconds (clipped to -90..120)",
    ylabel="clips",
)
ax.legend(loc="upper right")
plt.show()
summary["clip_timing"]

## 3. Spike alignment

Spikes are 10 s windows with the features' VOD z-score at or above 3 (`chat_msg_rate_z`,
`audio_rms_max_z`), 60 s apart. *Lift* is the share of spikes within 15 s of a moment divided
by the share of all windows that are (moments cover much of a busy VOD). Chat spikes are
shifted back by the pooled chat delay unless marked "unshifted". Means over VODs, 95 % intervals.

In [ ]:
main = summary["alignment"]["all"]
names = [
    ("chat_lift", "chat (shifted)"),
    ("chat_lift_unshifted", "chat (unshifted)"),
    ("audio_lift", "audio"),
    ("chat_audio_lift", "chat near audio"),
]
fig, ax = plt.subplots(figsize=(7, 3.2))
for i, (key, _label) in enumerate(names):
    m = main[key]
    ax.bar(i, m["mean"], width=0.6, color=SERIES[0], edgecolor=SURFACE, linewidth=2)
    ax.errorbar(
        i,
        m["mean"],
        yerr=[[m["mean"] - m["low"]], [m["high"] - m["mean"]]],
        color=INK_2,
        capsize=4,
        lw=1,
    )
    ax.annotate(
        f"{m['mean']:.2f}",
        (i, m["high"]),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        color=INK,
    )
ax.axhline(1, color=INK_2, lw=1, ls="--")
ax.annotate(
    "chance",
    (len(names) - 0.5, 1),
    xytext=(0, 3),
    textcoords="offset points",
    ha="right",
    fontsize=8,
    color=INK_2,
)
ax.set_xticks(range(len(names)), [label for _, label in names])
ax.grid(axis="x", visible=False)
ax.set_ylim(0, max(main[k]["high"] for k, _ in names) * 1.18)
ax.set(title="Spikes near moments, relative to chance (z >= 3, 10 s windows)", ylabel="lift")
plt.show()
pl.DataFrame([{"metric": k, **(v or {})} for k, v in main.items()])

In [ ]:
sens = result.sensitivity.filter(pl.col("group") == "all")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for z, color in zip(config.sensitivity_z, SERIES, strict=False):
    one = sens.filter(pl.col("z") == z).sort("window_s")
    axes[0].plot(
        one["window_s"],
        one["chat_lift"],
        "o-",
        color=color,
        ms=6,
        mec=SURFACE,
        mew=2,
        label=f"z >= {z:g}",
    )
    axes[1].plot(
        one["window_s"],
        one["top_with_neither"],
        "o-",
        color=color,
        ms=6,
        mec=SURFACE,
        mew=2,
        label=f"z >= {z:g}",
    )
axes[0].set(title="Chat-spike lift by window size", xlabel="window (s)", ylabel="lift")
axes[1].set(title="Top moments with no chat or audio spike", xlabel="window (s)", ylabel="share")
axes[0].legend(loc="upper left")
plt.show()
sens.select(
    "window_s",
    "z",
    "chat_spikes_per_hour",
    "audio_spikes_per_hour",
    "chat_lift",
    "audio_lift",
    "top_with_chat",
    "top_with_audio",
    "top_with_neither",
)

## 4. Padding inputs

For each VOD's 5 most-viewed clips: where the loudest second (searched 30 s either side) and
the delay-shifted chat peak (30 s before, 60 s after) fall, measured from the clip's reported
start. The dashed lines mark the start and the typical clip end.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
bins = np.arange(-45, 91, 5)
for ax, column, color, title in (
    (axes[0], "loud_from_start_s", LOUD, "Loudest second"),
    (axes[1], "chat_from_start_s", CHAT, "Chat peak, delay-shifted"),
):
    ax.hist(result.padding[column], bins=bins, color=color, edgecolor=SURFACE, linewidth=2)
    ax.grid(axis="x", visible=False)
    mark_zero(ax, "reported start")
    ax.axvline(float(result.padding["clip_duration_s"].median()), color=MUTED, lw=1, ls="--")
    ax.set(
        title=f"{title}, top clips (n = {result.padding.height})",
        xlabel="seconds from the reported start",
    )
axes[0].set_ylabel("clips")
plt.show()
pl.DataFrame([{"signal": s, **summary["padding"]["all"][s]} for s in ("loud", "chat")])